# Install Required Libraries

In [ ]:
!pip install unsloth transformers peft trl bitsandbytes wandb torch accelerate datasets jinja2

# Set Up Colab Environment

In [ ]:
import torch
from google.colab import drive
drive.mount('/content/drive')

# Check GPU
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Set random seed for reproducibility
torch.manual_seed(42)

# Load the Base Model

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Choose any! We auto support RoPE Scaling internally!
dtype = None  # None for auto detection. Float16 for Tesla T4, V100, BFloat16 for Ampere+
load_in_4bit = True  # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

# Prepare the Dataset

In [ ]:
from datasets import load_dataset
from jinja2 import Template

# Load dataset (example: OpenThoughts)
dataset = load_dataset("Bespoke-Stratos/OpenThoughts", split="train")

# Load chat template
with open('/content/drive/MyDrive/sheikh-max/templates/sheikh_chat_template.jinja', 'r') as f:
    template_str = f.read()
template = Template(template_str)

# Function to format messages
def format_example(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["response"]}
    ]
    return {"text": template.render(messages=messages, bos_token=tokenizer.bos_token, eos_token=tokenizer.eos_token)}

# Apply formatting
dataset = dataset.map(format_example)

# Tokenize
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=max_seq_length)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Fine-Tune the Model

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,  # Supports any, but = 0 is optimized
    bias="none",  # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
    random_state=3407,
    use_rslora=False,  # We support rank stabilized LoRA
    loftq_config=None,  # And LoftQ
)

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,  # Set num_train_epochs=1 for full training runs
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="wandb",  # Use this for WandB etc
    ),
)

trainer.train()

# Test the Model

In [ ]:
# Enable faster inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "Write a Python function to calculate factorial."}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    use_cache=True,
    temperature=0.7,
    min_p=0.1
)

decoded = tokenizer.batch_decode(outputs)[0]
print(decoded)

# Check for <think> tags
if "<think>" in decoded and "</think>" in decoded:
    print("Interleaved thinking detected!")
else:
    print("No interleaved thinking found.")